# Seven-page pre/post-merge comparator

This diagnostic records only CSV/JSON metadata for Page003, Page004, Page005, Page008, Page036, Page040 and Page049. It does not save page arrays, masks or inpaint results.

Run sections A and B in Notebook 51R2, repeat them in Notebook 53R2 after changing `TRACE_SIDE`, then run section C once.


## A. Install seven-page trace


In [ ]:
# ============================================================
# SEVEN-PAGE PRE/POST-MERGE TRACE
# Run in Notebook 51R2, then in Notebook 53R2.
#
# Place this cell immediately before the original reconstruction
# cell. It writes CSV/JSON only. No source, mask or result arrays
# are saved.
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import inspect
import json
import platform
import sys

import cv2
import numpy as np
import pandas as pd

TRACE_SIDE = '51R2'  # CHANGE TO '53R2' IN NOTEBOOK 53R2
ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')
TRACE_ROOT = ROOT / 'stage_f_seven_page_merge_trace' / TRACE_SIDE
TRACE_ROOT.mkdir(parents=True, exist_ok=True)

FAILING_PAGES = {
    'Page003', 'Page004', 'Page005', 'Page008',
    'Page036', 'Page040', 'Page049',
}

# These are diagnostic labels, not assumptions about arm naming.
ARM_ALIASES = {'M5', 'M5_REMOVE'}

TRACE_ROWS = []
INPAINT_ROWS = []
TRACE_EVENT_NUMBER = 0
INPAINT_CALL_NUMBER = 0


def semantic_hash(array):
    array = np.ascontiguousarray(array)
    h = hashlib.sha256()
    h.update(str(array.dtype).encode())
    h.update(str(array.shape).encode())
    h.update(array.tobytes())
    return h.hexdigest()


def scalar(value):
    if isinstance(value, (str, int, float, bool, Path, np.integer, np.floating, np.bool_)):
        return str(value)
    return None


def stack_context():
    """Recover page/region/bbox labels without retaining arrays."""
    context = {}
    wanted = (
        'page', 'page_id', 'page_name', 'region', 'region_id', 'rid',
        'arm', 'restorer', 'method', 'x0', 'y0', 'x1', 'y1',
        'margin', 'min_area', 'CROP_MARGIN', 'MIN_COMPONENT_AREA',
    )
    frames = inspect.stack()
    try:
        for frame_info in frames[2:30]:
            loc = frame_info.frame.f_locals
            for name in wanted:
                if name in context or name not in loc:
                    continue
                value = scalar(loc[name])
                if value is not None:
                    context[name] = value
    finally:
        del frames
    return context


def context_page(context):
    return context.get('page') or context.get('page_id') or context.get('page_name')


def normalise_box(box):
    return tuple(int(x) for x in box)


def boxes_overlap(a, b):
    ax0, ay0, ax1, ay1 = a
    bx0, by0, bx1, by1 = b
    return not (ax1 < bx0 or bx1 < ax0 or ay1 < by0 or by1 < ay0)


def merge_boxes_with_passes(boxes):
    """Canonical repeated overlap merge, with every pass retained as rows."""
    current = [normalise_box(x) for x in boxes]
    passes = []
    pass_number = 0

    while True:
        pass_number += 1
        changed = False
        work = list(current)
        merged = []

        while work:
            ax0, ay0, ax1, ay1 = work.pop()
            rest = []
            absorbed = 1

            for bx0, by0, bx1, by1 in work:
                b = (bx0, by0, bx1, by1)
                a = (ax0, ay0, ax1, ay1)
                if boxes_overlap(a, b):
                    ax0 = min(ax0, bx0)
                    ay0 = min(ay0, by0)
                    ax1 = max(ax1, bx1)
                    ay1 = max(ay1, by1)
                    absorbed += 1
                    changed = True
                else:
                    rest.append(b)

            work = rest
            merged.append((ax0, ay0, ax1, ay1))
            passes.append({
                'merge_pass': pass_number,
                'box_in_pass': len(merged),
                'x0': ax0, 'y0': ay0, 'x1': ax1, 'y1': ay1,
                'absorbed_boxes': absorbed,
            })

        current = merged
        if not changed:
            break

    return current, passes


def trace_mask_decomposition(mask, page, label, margin=48, min_area=1):
    """
    Independently records connected components, padded boxes, every
    merge pass and final boxes for one exact mask. CSV metadata only.
    """
    global TRACE_EVENT_NUMBER
    TRACE_EVENT_NUMBER += 1
    event = TRACE_EVENT_NUMBER

    mask_bool = np.asarray(mask) > 0
    n, labels, stats, _ = cv2.connectedComponentsWithStats(
        mask_bool.astype(np.uint8), 8
    )
    height, width = mask_bool.shape
    initial = []

    for component_id in range(1, n):
        x, y, w, h, area = [int(v) for v in stats[component_id]]
        retained = area >= int(min_area)
        if retained:
            box = (
                max(0, x - int(margin)),
                max(0, y - int(margin)),
                min(width, x + w + int(margin)),
                min(height, y + h + int(margin)),
            )
            initial.append(box)
        else:
            box = (None, None, None, None)

        TRACE_ROWS.append({
            'side': TRACE_SIDE, 'event': event, 'page': page,
            'label': label, 'stage': 'PRE_COMPONENT',
            'component_id': component_id,
            'component_x': x, 'component_y': y,
            'component_w': w, 'component_h': h,
            'component_area': area, 'retained': retained,
            'margin': int(margin), 'min_area': int(min_area),
            'x0': box[0], 'y0': box[1], 'x1': box[2], 'y1': box[3],
            'mask_pixels': int(mask_bool.sum()),
            'mask_boolean_sha256': semantic_hash(mask_bool.astype(np.uint8)),
        })

    for ordinal, box in enumerate(initial, 1):
        TRACE_ROWS.append({
            'side': TRACE_SIDE, 'event': event, 'page': page,
            'label': label, 'stage': 'PADDED_PRE_MERGE',
            'box_ordinal': ordinal, 'margin': int(margin),
            'min_area': int(min_area),
            'x0': box[0], 'y0': box[1], 'x1': box[2], 'y1': box[3],
            'mask_pixels': int(mask_bool.sum()),
            'mask_boolean_sha256': semantic_hash(mask_bool.astype(np.uint8)),
        })

    final_boxes, merge_pass_rows = merge_boxes_with_passes(initial)

    for row in merge_pass_rows:
        TRACE_ROWS.append({
            'side': TRACE_SIDE, 'event': event, 'page': page,
            'label': label, 'stage': 'MERGE_PASS',
            'margin': int(margin), 'min_area': int(min_area),
            'mask_pixels': int(mask_bool.sum()),
            'mask_boolean_sha256': semantic_hash(mask_bool.astype(np.uint8)),
            **row,
        })

    for ordinal, box in enumerate(final_boxes, 1):
        x0, y0, x1, y1 = box
        local_mask = mask_bool[y0:y1, x0:x1]
        TRACE_ROWS.append({
            'side': TRACE_SIDE, 'event': event, 'page': page,
            'label': label, 'stage': 'POST_MERGE',
            'box_ordinal': ordinal, 'margin': int(margin),
            'min_area': int(min_area),
            'x0': x0, 'y0': y0, 'x1': x1, 'y1': y1,
            'box_width': x1 - x0, 'box_height': y1 - y0,
            'box_mask_pixels': int(local_mask.sum()),
            'box_mask_boolean_sha256': semantic_hash(local_mask.astype(np.uint8)),
            'mask_pixels': int(mask_bool.sum()),
            'mask_boolean_sha256': semantic_hash(mask_bool.astype(np.uint8)),
        })

    return final_boxes


# ------------------------------------------------------------
# Lightweight cv2.inpaint capture
# ------------------------------------------------------------

if hasattr(cv2, '_stage_f_merge_trace_original_inpaint'):
    ORIGINAL_INPAINT = cv2._stage_f_merge_trace_original_inpaint
else:
    ORIGINAL_INPAINT = cv2.inpaint
    cv2._stage_f_merge_trace_original_inpaint = ORIGINAL_INPAINT


def captured_inpaint(source, mask, radius, flags):
    global INPAINT_CALL_NUMBER
    result = ORIGINAL_INPAINT(source, mask, radius, flags)
    context = stack_context()
    page = context_page(context)

    # Fail closed on storage volume: retain only the seven target pages.
    if page not in FAILING_PAGES:
        return result

    INPAINT_CALL_NUMBER += 1
    mask_bool = np.asarray(mask) > 0
    ys, xs = np.where(mask_bool)
    local_box = None if not len(xs) else (
        int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1
    )

    # If crop-origin variables are present, convert the local nonzero box
    # to page coordinates. Otherwise the page box remains unknown.
    page_box = None
    if local_box is not None and context.get('x0') is not None and context.get('y0') is not None:
        ox, oy = int(context['x0']), int(context['y0'])
        page_box = (
            ox + local_box[0], oy + local_box[1],
            ox + local_box[2], oy + local_box[3],
        )
    elif local_box is not None and np.asarray(source).ndim == 2:
        # Full-page calls frequently have no explicit crop origin.
        page_box = local_box

    INPAINT_ROWS.append({
        'side': TRACE_SIDE,
        'call_number': INPAINT_CALL_NUMBER,
        'page': page,
        'region_id': context.get('region_id') or context.get('region') or context.get('rid'),
        'arm': context.get('arm'),
        'restorer': context.get('restorer') or context.get('method'),
        'source_height': int(np.asarray(source).shape[0]),
        'source_width': int(np.asarray(source).shape[1]),
        'mask_pixels': int(mask_bool.sum()),
        'mask_boolean_sha256': semantic_hash(mask_bool.astype(np.uint8)),
        'radius': float(radius),
        'flags': int(flags),
        'local_x0': None if local_box is None else local_box[0],
        'local_y0': None if local_box is None else local_box[1],
        'local_x1': None if local_box is None else local_box[2],
        'local_y1': None if local_box is None else local_box[3],
        'page_x0': None if page_box is None else page_box[0],
        'page_y0': None if page_box is None else page_box[1],
        'page_x1': None if page_box is None else page_box[2],
        'page_y1': None if page_box is None else page_box[3],
        'context_json': json.dumps(context, sort_keys=True),
    })
    return result


cv2.inpaint = captured_inpaint

print('Seven-page trace installed for', TRACE_SIDE)
print('Run the ORIGINAL reconstruction cell now.')
print('No image arrays will be written.')


## B. Finalise one side


In [ ]:
# ============================================================
# FINALISE THIS SIDE AFTER THE ORIGINAL RECONSTRUCTION CELL
# ============================================================

cv2.inpaint = ORIGINAL_INPAINT

# # Optional direct decomposition trace
# # If your notebook exposes exact page-level masks in a dictionary,
# # call trace_mask_decomposition once per target page before finalising:
# #
# # for page in sorted(FAILING_PAGES):
# #     trace_mask_decomposition(
# #         exact_page_masks[page], page, 'PAGE_M5', margin=48, min_area=1
# #     )

trace_columns = [
    'side','event','page','label','stage','component_id','component_x',
    'component_y','component_w','component_h','component_area','retained',
    'box_ordinal','merge_pass','box_in_pass','absorbed_boxes','margin',
    'min_area','x0','y0','x1','y1','box_width','box_height',
    'box_mask_pixels','box_mask_boolean_sha256','mask_pixels',
    'mask_boolean_sha256'
]
call_columns = [
    'side','call_number','page','region_id','arm','restorer',
    'source_height','source_width','mask_pixels','mask_boolean_sha256',
    'radius','flags','local_x0','local_y0','local_x1','local_y1',
    'page_x0','page_y0','page_x1','page_y1','context_json'
]

trace_df = pd.DataFrame(TRACE_ROWS, columns=trace_columns)
call_df = pd.DataFrame(INPAINT_ROWS, columns=call_columns)

trace_df.to_csv(TRACE_ROOT / 'pre_post_merge_trace.csv', index=False)
call_df.to_csv(TRACE_ROOT / 'inpaint_call_trace.csv', index=False)

summary = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'side': TRACE_SIDE,
    'failing_pages': sorted(FAILING_PAGES),
    'decomposition_rows': int(len(trace_df)),
    'captured_target_inpaint_calls': int(len(call_df)),
    'pages_in_call_trace': sorted(call_df.page.dropna().astype(str).unique().tolist()) if len(call_df) else [],
    'image_arrays_saved': False,
    'python': sys.version,
    'opencv': cv2.__version__,
    'numpy': np.__version__,
    'platform': platform.platform(),
}
(TRACE_ROOT / 'capture_summary.json').write_text(json.dumps(summary, indent=2))

print(json.dumps(summary, indent=2))
print('Output:', TRACE_ROOT)
if call_df.empty:
    raise RuntimeError(
        'No target-page inpaint calls were captured. Page identity was not '
        'recoverable from the call stack, or checkpoints were reused.'
    )
display(call_df.groupby('page', dropna=False).size().rename('calls').reset_index())


## C. Compare 51R2 and 53R2


In [ ]:
# ============================================================
# SEVEN-PAGE PRE/POST-MERGE COMPARATOR
# Run after both 51R2 and 53R2 captures are complete.
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import json
import cv2
import numpy as np
import pandas as pd

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')
BASE = ROOT / 'stage_f_seven_page_merge_trace'
LEFT = BASE / '51R2'
RIGHT = BASE / '53R2'
OUT = BASE / ('comparison_' + datetime.now().strftime('%Y%m%d_%H%M%S'))
OUT.mkdir(parents=True, exist_ok=False)

FAILING_PAGES = [
    'Page003','Page004','Page005','Page008','Page036','Page040','Page049'
]


def require(condition, message):
    if not condition:
        raise RuntimeError(message)


def read_required(path):
    require(path.exists(), f'Missing required file: {path}')
    return pd.read_csv(path, keep_default_na=False)


left_calls = read_required(LEFT / 'inpaint_call_trace.csv')
right_calls = read_required(RIGHT / 'inpaint_call_trace.csv')

# Keep only the affected pages and Telea calls.
left_calls = left_calls[left_calls.page.isin(FAILING_PAGES)].copy()
right_calls = right_calls[right_calls.page.isin(FAILING_PAGES)].copy()
left_calls = left_calls[left_calls.flags.astype(int) == int(cv2.INPAINT_TELEA)].copy()
right_calls = right_calls[right_calls.flags.astype(int) == int(cv2.INPAINT_TELEA)].copy()

require(len(left_calls), 'No 51R2 target-page Telea rows.')
require(len(right_calls), 'No 53R2 target-page Telea rows.')

# A semantic call identity does not depend on M5 versus M5_REMOVE naming.
def call_signature(row):
    return (
        str(row.mask_boolean_sha256),
        int(row.mask_pixels),
        int(row.source_height),
        int(row.source_width),
        float(row.radius),
        int(row.flags),
    )

summary_rows = []
detail_rows = []

for page in FAILING_PAGES:
    l = left_calls[left_calls.page == page].copy()
    r = right_calls[right_calls.page == page].copy()
    lc = Counter(call_signature(row) for _, row in l.iterrows())
    rc = Counter(call_signature(row) for _, row in r.iterrows())
    common = lc & rc
    left_only = lc - rc
    right_only = rc - lc

    summary_rows.append({
        'page': page,
        'calls_51r2': int(sum(lc.values())),
        'calls_53r2': int(sum(rc.values())),
        'exact_signature_calls_shared': int(sum(common.values())),
        'calls_only_51r2': int(sum(left_only.values())),
        'calls_only_53r2': int(sum(right_only.values())),
        'call_multiset_exact': lc == rc,
    })

    for status, counter in [
        ('SHARED', common), ('ONLY_51R2', left_only), ('ONLY_53R2', right_only)
    ]:
        for signature, count in counter.items():
            mask_hash, pixels, sh, sw, radius, flags = signature
            detail_rows.append({
                'page': page, 'status': status, 'multiplicity': int(count),
                'mask_boolean_sha256': mask_hash, 'mask_pixels': pixels,
                'source_height': sh, 'source_width': sw,
                'radius': radius, 'flags': flags,
            })

summary_df = pd.DataFrame(summary_rows)
detail_df = pd.DataFrame(detail_rows)
summary_df.to_csv(OUT / 'call_population_by_page.csv', index=False)
detail_df.to_csv(OUT / 'call_signature_differences.csv', index=False)

# Compare separately captured decomposition tables when they are populated.
left_merge_path = LEFT / 'pre_post_merge_trace.csv'
right_merge_path = RIGHT / 'pre_post_merge_trace.csv'
left_merge = pd.read_csv(left_merge_path, keep_default_na=False) if left_merge_path.exists() else pd.DataFrame()
right_merge = pd.read_csv(right_merge_path, keep_default_na=False) if right_merge_path.exists() else pd.DataFrame()

merge_summary_rows = []
merge_detail_rows = []

if len(left_merge) and len(right_merge):
    for page in FAILING_PAGES:
        for stage in ['PRE_COMPONENT','PADDED_PRE_MERGE','POST_MERGE']:
            l = left_merge[(left_merge.page == page) & (left_merge.stage == stage)].copy()
            r = right_merge[(right_merge.page == page) & (right_merge.stage == stage)].copy()

            if stage == 'PRE_COMPONENT':
                cols = ['component_x','component_y','component_w','component_h','component_area','retained']
            else:
                cols = ['x0','y0','x1','y1']

            def tuples(df):
                if df.empty:
                    return []
                return sorted(tuple(str(row[c]) for c in cols) for _, row in df.iterrows())

            lt, rt = tuples(l), tuples(r)
            merge_summary_rows.append({
                'page': page, 'stage': stage,
                'rows_51r2': len(lt), 'rows_53r2': len(rt),
                'exact': lt == rt,
            })

            only_l = Counter(lt) - Counter(rt)
            only_r = Counter(rt) - Counter(lt)
            for status, counter in [('ONLY_51R2', only_l), ('ONLY_53R2', only_r)]:
                for values, count in counter.items():
                    row = {'page': page, 'stage': stage, 'status': status, 'multiplicity': count}
                    row.update({c: v for c, v in zip(cols, values)})
                    merge_detail_rows.append(row)

merge_summary_df = pd.DataFrame(merge_summary_rows)
merge_detail_df = pd.DataFrame(merge_detail_rows)
merge_summary_df.to_csv(OUT / 'pre_post_merge_summary.csv', index=False)
merge_detail_df.to_csv(OUT / 'pre_post_merge_differences.csv', index=False)

# First divergence classification.
if len(merge_summary_df) and not merge_summary_df.empty:
    failing = merge_summary_df[~merge_summary_df.exact]
    if len(failing):
        first_stage_order = {'PRE_COMPONENT': 1, 'PADDED_PRE_MERGE': 2, 'POST_MERGE': 3}
        first = failing.assign(order=failing.stage.map(first_stage_order)).sort_values(['order','page']).iloc[0]
        diagnosis = {
            'PRE_COMPONENT': 'CONNECTED_COMPONENT_POPULATION_DIFFERS',
            'PADDED_PRE_MERGE': 'PADDING_OR_COMPONENT_FILTER_DIFFERS',
            'POST_MERGE': 'OVERLAP_MERGE_DIFFERS',
        }[first.stage]
    elif summary_df.call_multiset_exact.all():
        diagnosis = 'DECOMPOSITION_AND_CALL_POPULATION_EXACT'
    else:
        diagnosis = 'DECOMPOSITION_TRACE_EXACT_BUT_INPAINT_CALL_POPULATION_DIFFERS'
else:
    if summary_df.call_multiset_exact.all():
        diagnosis = 'INPAINT_CALL_POPULATION_EXACT_NO_SEPARATE_MERGE_TRACE'
    else:
        diagnosis = 'INPAINT_CALL_POPULATION_DIFFERS_MERGE_TRACE_NOT_POPULATED'

verdict = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'experiment': 'Seven-page pre/post-merge and Telea-call population comparator',
    'pages': FAILING_PAGES,
    'calls_51r2': int(len(left_calls)),
    'calls_53r2': int(len(right_calls)),
    'all_page_call_multisets_exact': bool(summary_df.call_multiset_exact.all()),
    'separate_pre_post_merge_trace_available': bool(len(left_merge) and len(right_merge)),
    'diagnosis': diagnosis,
    'image_arrays_loaded_or_saved_by_comparator': False,
}
(OUT / 'seven_page_merge_comparator_verdict.json').write_text(json.dumps(verdict, indent=2))

print(json.dumps(verdict, indent=2))
print('\nCall population by page:')
display(summary_df)
if len(merge_summary_df):
    print('\nPre/post-merge summary:')
    display(merge_summary_df)
print('\nOutput:', OUT)
